# Qwen 3.5 27B FP8 — test offline avec kernel local

Correction du notebook précédent : `local_fp8_kernel` n'est jamais supposé exister avant d'avoir été construit.
Le notebook diagnostique d'abord le kernel local, le charge, puis patche Transformers avant le chargement/génération du modèle.

**Exécuter les cellules dans l'ordre.**

## 1. Environnement

In [ ]:
import os, sys, torch, transformers
os.environ["TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"] = "1"

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU         :", torch.cuda.get_device_name(0))

## 2. Modèle local

In [ ]:
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.5-27B-FP8/main"
print("MODEL_PATH:", MODEL_PATH)
print("Existe:", os.path.exists(MODEL_PATH))
assert os.path.isdir(MODEL_PATH), MODEL_PATH
print("Fichiers:", os.listdir(MODEL_PATH)[:30])

## 3. Module FP8 Transformers

In [ ]:
import transformers.integrations.finegrained_fp8 as fp8
import inspect

print("Module:", fp8)
print("load_finegrained_fp8_kernel:", fp8.load_finegrained_fp8_kernel)
print("lazy_load_kernel:", fp8.lazy_load_kernel)
print("Source loader:")
print(inspect.getsource(fp8._load_finegrained_fp8_kernel))

## 4. Localiser le kernel FP8 déjà présent sur Domino

Cette cellule ne contacte pas Hugging Face. Elle cherche uniquement des fichiers locaux contenant `finegrained` / `fp8`.

In [ ]:
from pathlib import Path

roots = [
    Path.home() / ".cache",
    Path("/domino"),
    Path("/tmp"),
]

hits = []
for root in roots:
    if not root.exists():
        continue
    try:
        for p in root.rglob("*finegrained*fp8*"):
            hits.append(p)
            if len(hits) >= 100:
                break
    except (PermissionError, OSError):
        pass
    if len(hits) >= 100:
        break

print("Résultats:", len(hits))
for p in hits[:50]:
    print(p)

## 5. Indiquer le dossier du kernel local

Si la cellule 4 affiche clairement le dossier du kernel `finegrained-fp8`, recopiez-le dans `LOCAL_KERNEL_DIR`.

Si vous aviez déjà installé/copier ce kernel ailleurs dans Domino, mettez directement son chemin ici.

In [ ]:
LOCAL_KERNEL_DIR = None

# EXEMPLE SEULEMENT :
# LOCAL_KERNEL_DIR = "/domino/.../finegrained-fp8"

print("LOCAL_KERNEL_DIR =", LOCAL_KERNEL_DIR)

## 6. Construire `local_fp8_kernel`

In [ ]:
import importlib.util
from pathlib import Path

if not LOCAL_KERNEL_DIR:
    raise RuntimeError(
        "Renseignez LOCAL_KERNEL_DIR avec le chemin affiché à la cellule 4. "
        "On ne lance volontairement aucun téléchargement Internet."
    )

root = Path(LOCAL_KERNEL_DIR)
assert root.is_dir(), f"Dossier introuvable: {root}"

local_fp8_kernel = None
errors = []

for py in root.rglob("*.py"):
    if py.name.startswith("__"):
        continue
    try:
        spec = importlib.util.spec_from_file_location(
            "domino_local_finegrained_fp8", str(py)
        )
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        if all(hasattr(mod, name) for name in
               ("matmul", "matmul_batched", "grouped_matmul")):
            local_fp8_kernel = mod
            print("Kernel trouvé dans:", py)
            break
    except Exception as e:
        errors.append((str(py), repr(e)))

if local_fp8_kernel is None:
    print("Premières erreurs d'import:")
    for x in errors[:10]:
        print(x)
    raise RuntimeError(
        "Le dossier existe mais aucun module exposant matmul, "
        "matmul_batched et grouped_matmul n'a été trouvé."
    )

print("local_fp8_kernel =", local_fp8_kernel)
print("matmul =", local_fp8_kernel.matmul)
print("matmul_batched =", local_fp8_kernel.matmul_batched)
print("grouped_matmul =", local_fp8_kernel.grouped_matmul)

## 7. Patch complet du loader FP8

In [ ]:
def local_loader(*args, **kwargs):
    return local_fp8_kernel

# Les différentes références susceptibles d'être utilisées
fp8.lazy_load_kernel = local_loader
fp8._load_finegrained_fp8_kernel = local_loader
fp8.load_finegrained_fp8_kernel = local_loader

print("Patch installé.")
print("lazy_load_kernel:", fp8.lazy_load_kernel)
print("_load_finegrained_fp8_kernel:", fp8._load_finegrained_fp8_kernel)
print("load_finegrained_fp8_kernel:", fp8.load_finegrained_fp8_kernel)

## 8. Test du chemin réellement utilisé

In [ ]:
k = fp8.load_finegrained_fp8_kernel()

assert k is local_fp8_kernel
assert callable(k.matmul)
assert callable(k.matmul_batched)
assert callable(k.grouped_matmul)

print("LOADER FP8 LOCAL : OK")
print(k)

## 9. Charger le modèle local

On ne passe à cette étape que si la cellule 8 affiche `LOADER FP8 LOCAL : OK`.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto",
)
model.eval()

print("Processor OK")
print("Model OK:", type(model))
print("Device:", model.device)

## 10. Test minimal de génération

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par TEST OK"}]
}]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(text=[text], return_tensors="pt")
device = next(model.parameters()).device
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

print("Tokens entrée:", inputs["input_ids"].shape[-1])

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

generated = outputs[:, inputs["input_ids"].shape[-1]:]
response = processor.batch_decode(generated, skip_special_tokens=True)[0]

print("=" * 60)
print("REPONSE QWEN")
print("=" * 60)
print(response)
print("=" * 60)